# 03 — Contaminated Gaussian simulations

This replaces the three separate epsilon-specific notebooks.
It runs the same analysis for
\(\epsilon\in\{0.05,0.10,0.15\}\) with contamination scale
\(	au=9\).

In [ ]:
using Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)

    while true
        if isdir(joinpath(dir, "src"))
            return dir
        end

        parent = dirname(dir)
        parent == dir && error(
            "Could not find the repository root. " *
            "Run this notebook somewhere inside the mad-epb repository."
        )

        dir = parent
    end
end

REPO_ROOT = find_repo_root()

project_file = joinpath(REPO_ROOT, "Project.toml")
isfile(project_file) && Pkg.activate(REPO_ROOT)

using Distributions
using Random
using Statistics
using MultipleTesting
using Empirikos
using Hypatia
using LinearAlgebra
using JLD2

include(joinpath(REPO_ROOT, "src", "mad_distribution.jl"))
include(joinpath(REPO_ROOT, "src", "mad_tests.jl"))
include(joinpath(REPO_ROOT, "src", "mad_epb.jl"))
include(joinpath(REPO_ROOT, "src", "simulation_utils.jl"))

RESULTS_DIR = joinpath(REPO_ROOT, "results", "simulation_outputs")
mkpath(RESULTS_DIR)

println("Repository root: ", REPO_ROOT)
println("Results directory: ", RESULTS_DIR)

In [ ]:
# ============================================================
# Existing variance-based EPB benchmarks
# ============================================================

function variance_epb_pvalues(
    Y;
    prior::Symbol = :limma,
    alpha::Float64 = 0.05
)
    n = length(first(Y))

    if any(length(y) != n for y in Y)
        error("A common within-gene sample size is required.")
    end

    ν = n - 1

    beta_hat = [mean(y) for y in Y]
    s_hat = [std(y, corrected = true) for y in Y]
    se_hat_squared = (s_hat .^ 2) ./ n

    samples = [
        Empirikos.NormalChiSquareSample(
            beta_hat[i],
            se_hat_squared[i],
            ν
        )
        for i in eachindex(Y)
    ]

    test =
        prior == :limma ?
        Empirikos.EmpiricalPartiallyBayesTTest(
            α = alpha,
            prior = Empirikos.Limma(),
            solver = Hypatia.Optimizer
        ) :
        prior == :npmle ?
        Empirikos.EmpiricalPartiallyBayesTTest(
            α = alpha,
            solver = Hypatia.Optimizer
        ) :
        error("prior must be :limma or :npmle")

    result = fit(test, samples)

    return (
        pvalues = collect(result.pvalue),
        reject = collect(result.rj_idx),
        fit = result
    )
end


# ============================================================
# Summary helpers
# ============================================================

function empty_summary()
    Dict(
        "n" => Int[],
        "TP" => Float64[],
        "FP" => Float64[],
        "Power" => Float64[],
        "FDR" => Float64[],
        "TP_top100" => Float64[]
    )
end

function summarize_pvalue_store(
    pvalue_store,
    is_DE_store,
    n_values,
    R;
    alpha::Float64 = 0.05,
    topk::Int = 100
)
    summary = empty_summary()

    for n in n_values
        tp_vals = Float64[]
        fp_vals = Float64[]
        power_vals = Float64[]
        fdr_vals = Float64[]
        top_vals = Float64[]

        for r in 1:R
            key = (n, r)

            if !haskey(pvalue_store, key) || !haskey(is_DE_store, key)
                continue
            end

            metrics = evaluate_pvalues(
                pvalue_store[key],
                is_DE_store[key];
                alpha = alpha,
                topk = topk
            )

            push!(tp_vals, metrics.TP)
            push!(fp_vals, metrics.FP)
            push!(power_vals, metrics.Power)
            push!(fdr_vals, metrics.FDP)
            push!(top_vals, metrics.TP_topk)
        end

        isempty(power_vals) && continue

        push!(summary["n"], n)
        push!(summary["TP"], mean(tp_vals))
        push!(summary["FP"], mean(fp_vals))
        push!(summary["Power"], mean(power_vals))
        push!(summary["FDR"], mean(fdr_vals))
        push!(summary["TP_top100"], mean(top_vals))
    end

    return summary
end


function print_summary(name, summary)
    println("\n========== $(uppercase(name)) ==========")

    for i in eachindex(summary["n"])
        println(
            "n=$(summary["n"][i]) | " *
            "Power=$(round(summary["Power"][i], digits=4)) | " *
            "FDR=$(round(summary["FDR"][i], digits=4)) | " *
            "TP@100=$(round(summary["TP_top100"][i], digits=2))"
        )
    end
end


# ============================================================
# Run all six procedures on one collection of datasets
# ============================================================

function run_all_methods(
    datasets;
    n_values,
    R::Int,
    alpha::Float64 = 0.05,
    mad_null_K::Int = 400,
    mad_param_K::Int = 100,
    mad_param_max_iter::Int = 80,
    mad_np_grid::Int = 200
)
    method_names = [
        :ttest,
        :mad_test,
        :limma,
        :variance_np,
        :mad_parametric,
        :mad_np
    ]

    pvalue_stores = Dict(
        method => Dict{Tuple{Int,Int}, Vector{Float64}}()
        for method in method_names
    )

    mad_param_prior_store =
        Dict{Tuple{Int,Int}, NamedTuple}()

    is_DE_store = extract_is_DE_store(datasets)

    # The exact null quadrature depends only on n, so build it once.
    mad_quad_store = Dict(
        n => build_mad_null_quadrature(n; K = mad_null_K)
        for n in n_values
    )

    for n in n_values
        println("\n================================================")
        println("n = $n")
        println("================================================")

        for r in 1:R
            key = (n, r)
            Y = datasets[key].Y

            # ------------------------------------------------
            # 1. Ordinary t test
            # ------------------------------------------------
            try
                out = ttest_pvalues(Y)
                pvalue_stores[:ttest][key] =
                    Float64.(out.pvalues)
            catch err
                @warn "t-test failed for key=$key" exception=(err, catch_backtrace())
            end

            # ------------------------------------------------
            # 2. Non-EB MAD-normalized test
            # ------------------------------------------------
            try
                out = mad_test_pvalues(
                    Y;
                    quad = mad_quad_store[n]
                )

                pvalue_stores[:mad_test][key] =
                    Float64.(out.pvalues)
            catch err
                @warn "MAD test failed for key=$key" exception=(err, catch_backtrace())
            end

            # ------------------------------------------------
            # 3. Limma-style parametric variance EPB
            # ------------------------------------------------
            try
                out = variance_epb_pvalues(
                    Y;
                    prior = :limma,
                    alpha = alpha
                )

                pvalue_stores[:limma][key] =
                    Float64.(out.pvalues)
            catch err
                @warn "Limma failed for key=$key" exception=(err, catch_backtrace())
            end

            # ------------------------------------------------
            # 4. Nonparametric variance EPB
            # ------------------------------------------------
            try
                out = variance_epb_pvalues(
                    Y;
                    prior = :npmle,
                    alpha = alpha
                )

                pvalue_stores[:variance_np][key] =
                    Float64.(out.pvalues)
            catch err
                @warn "Variance NPMLE failed for key=$key" exception=(err, catch_backtrace())
            end

            # ------------------------------------------------
            # 5. Parametric MAD EPB
            # ------------------------------------------------
            try
                out = mad_parametric_pvalues(
                    Y;
                    K_quad = mad_param_K,
                    max_iter = mad_param_max_iter
                )

                pvalue_stores[:mad_parametric][key] =
                    Float64.(out.pvalues)

                mad_param_prior_store[key] = (
                    sigma0_sq_hat = out.fit.σ0²,
                    nu0_hat = out.fit.ν0
                )
            catch err
                @warn "Parametric MAD EPB failed for key=$key" exception=(err, catch_backtrace())
            end

            # ------------------------------------------------
            # 6. Nonparametric MAD EPB
            # ------------------------------------------------
            try
                out = mad_nonparametric_pvalues(
                    Y;
                    n_grid = mad_np_grid
                )

                pvalue_stores[:mad_np][key] =
                    Float64.(out.pvalues)
            catch err
                @warn "Nonparametric MAD EPB failed for key=$key" exception=(err, catch_backtrace())
            end

            if r % 10 == 0
                println("finished replicate $r / $R")
            end
        end
    end

    summaries = Dict(
        method => summarize_pvalue_store(
            pvalue_stores[method],
            is_DE_store,
            n_values,
            R;
            alpha = alpha,
            topk = 100
        )
        for method in method_names
    )

    for method in method_names
        print_summary(string(method), summaries[method])
    end

    return (
        pvalue_stores = pvalue_stores,
        summaries = summaries,
        is_DE_store = is_DE_store,
        mad_param_prior_store = mad_param_prior_store
    )
end

## Simulation settings

In [ ]:
# Paper simulation settings
n_values = [4, 5, 6, 8, 10, 20, 30, 40]

G = 1000
R = 100

p_signal = 0.10
beta_signal = 4.0

# Across-gene variance distribution
s0_sq_true = 1.0
d0_true = 4.0

alpha = 0.05

# For a quick smoke test before the full run, temporarily use:
# R = 2
# n_values = [4, 10]

## Generate, analyze, and save each contamination setting

In [ ]:
eps_values = [0.05, 0.10, 0.15]
tau = 9.0

function generate_contaminated_normal_datasets(
    n_values,
    G,
    R;
    eps,
    tau,
    p_signal,
    beta_signal,
    s0_sq_true,
    d0_true
)
    datasets = Dict{Tuple{Int,Int}, Any}()

    for n in n_values
        for r in 1:R
            rng = MersenneTwister(10_000 + 100n + r)

            is_DE = rand(rng, G) .< p_signal

            sigma2 = rand(
                rng,
                InverseGamma(
                    d0_true / 2,
                    d0_true * s0_sq_true / 2
                ),
                G
            )

            sigma = sqrt.(sigma2)

            Y = Vector{Vector{Float64}}(undef, G)

            @inbounds for g in 1:G
                beta_g =
                    is_DE[g] ?
                    beta_signal * sigma[g] / sqrt(n) :
                    0.0

                z = rand_mixture_normal_errors(
                    rng,
                    n;
                    eps = eps,
                    tau = tau
                )

                Y[g] = beta_g .+ sigma[g] .* z
            end

            datasets[(n, r)] = (
                Y = Y,
                is_DE = is_DE,
                sigma = sigma,
                sigma2 = sigma2
            )
        end
    end

    return datasets
end


results_by_eps = Dict{Float64, Any}()

for eps in eps_values
    println("\n\n############################################")
    println("Contaminated Gaussian: eps = $eps, tau = $tau")
    println("############################################")

    datasets = generate_contaminated_normal_datasets(
        n_values,
        G,
        R;
        eps = eps,
        tau = tau,
        p_signal = p_signal,
        beta_signal = beta_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true
    )

    results = run_all_methods(
        datasets;
        n_values = n_values,
        R = R,
        alpha = alpha
    )

    simulation_params = (
        n_values = n_values,
        G = G,
        R = R,
        p_signal = p_signal,
        beta_signal = beta_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true,
        error_dist = "Contaminated Gaussian",
        eps = eps,
        tau = tau
    )

    results_by_eps[eps] = (
        simulation_params = simulation_params,
        results = results
    )

    eps_label = replace(string(eps), "." => "p")
    outfile = joinpath(
        RESULTS_DIR,
        "contaminated_normal_eps_$(eps_label).jld2"
    )

    @save outfile simulation_params results

    println("Saved: ", outfile)

    # Free the large simulated dataset before the next epsilon.
    datasets = nothing
    GC.gc()
end